In [1]:
from langchain_openai import ChatOpenAI  
from langchain_core.messages import HumanMessage

LLM = ChatOpenAI(model="gpt-4o-mini")   # Initialize the chat model

LLM.invoke([HumanMessage(content="안녕, 잘 지냈어?")]) # Send a message to the model

AIMessage(content='안녕하세요! 저는 잘 지내고 있어요. 당신은 어떻게 지내고 계신가요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 15, 'total_tokens': 37, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_af5b2e4445', 'id': 'chatcmpl-Eg1mo5Y7wbd2s3zQ4BKw9ltDUugkv', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a19ffa-63e2-7101-80d0-e9848d375de6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 22, 'total_tokens': 37, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [2]:
from langchain_core.tools import tool
from datetime import datetime
import pytz

@tool       # @tool decorator uses langchain's tool system
def get_current_time(timezone: str, location: str) -> str:
    """ 현재 시각 반환 함수
    Args:
        timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함
        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 LLM 답변 생성에 사용됨
    """
    Time_zone = pytz.timezone(timezone)
    now = datetime.now(Time_zone).strftime("%Y-%m-%d %H:%M:%S")

    # Return format: 'Asia/Seoul (Seoul) 현재시각 2024-06-01 15:30:00'
    location_and_local_time = f'{timezone} ({location}) 현재시각 {now} '
     
    print(location_and_local_time)
    return location_and_local_time

In [3]:
# Append the tool to a list and a dictionary
tools = [get_current_time,]
tool_dict = {"get_current_time": get_current_time,}

# Bind the tools to the LLM 
LLM_with_tools = LLM.bind_tools(tools)

In [4]:
from langchain_core.messages import SystemMessage

# Create messages for the conversation
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools 사용 가능한 AI 어시스턴트야."),
    HumanMessage("인천은 지금 몇시야?"),
]

# Call the LLM_with_tools to generate a response
response = LLM_with_tools.invoke(messages)
messages.append(response)

# Print the conversation messages
print(messages)

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools 사용 가능한 AI 어시스턴트야.', additional_kwargs={}, response_metadata={}), HumanMessage(content='인천은 지금 몇시야?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 138, 'total_tokens': 161, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_fbcfd2e298', 'id': 'chatcmpl-Eg1mxIAbVjpakl8NdofJCfAfyE7h9', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a19ffa-8635-70eb-b78b-43516a48f5fe-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezo

In [5]:
for tool_call in response.tool_calls:
    # Select the tool in tool_dict based on the tool name
    selected_tool = tool_dict[tool_call["name"]] 
    print(tool_call["args"])                   # Arguments for the tool function
    tool_msg = selected_tool.invoke(tool_call) # Result from the tool function
    messages.append(tool_msg)

messages

{'timezone': 'Asia/Seoul', 'location': '인천'}
Asia/Seoul (인천) 현재시각 2026-11-03 14:56:04 


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools 사용 가능한 AI 어시스턴트야.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='인천은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 138, 'total_tokens': 161, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_fbcfd2e298', 'id': 'chatcmpl-Eg1mxIAbVjpakl8NdofJCfAfyE7h9', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a19ffa-8635-70eb-b78b-43516a48f5fe-0', tool_calls=[{'name': 'get_current_time', 'args': {'time

In [6]:
LLM_with_tools.invoke(messages)

AIMessage(content='인천의 현재 시각은 2026년 11월 3일 14시 56분입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 195, 'total_tokens': 221, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_fbcfd2e298', 'id': 'chatcmpl-Eg1n5Yv3hyVzmsKBidHOKSa24bjaa', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a19ffa-a894-7255-901b-d1ced1874fca-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 195, 'output_tokens': 26, 'total_tokens': 221, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [7]:
from pydantic import BaseModel, Field

class StockHistoryInput(BaseModel):
    ticker: str = Field(..., title="주식 코드", description="주식 코드 (예: TSLA, AAPL)")
    period: str = Field(..., title="기간", description="주식 데이터 조회 기간 (예: 1d, 1mo, 1y)")

In [8]:
import yfinance as yf

@tool
def get_yf_stock_history(stock_history_input: StockHistoryInput) -> str:
    """ 주식 종목의 가격 데이터를 조회하는 함수"""
    stock = yf.Ticker(stock_history_input.ticker)
    history = stock.history(period=stock_history_input.period)
    history_md = history.to_markdown() 

    return history_md

tools = [get_current_time, get_yf_stock_history]
tool_dict = {"get_current_time": get_current_time, "get_yf_stock_history": get_yf_stock_history}

LLM_with_tools = LLM.bind_tools(tools)

In [9]:
messages.append(HumanMessage("테슬라는 한달 전에 비해 주가가 올랐나 내렸나?"))

response = LLM_with_tools.invoke(messages)
print(response)
messages.append(response)

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 71, 'prompt_tokens': 289, 'total_tokens': 360, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_fbfbfe4157', 'id': 'chatcmpl-Eg1nErH1kksKlgiJXiRVoduRNHAMH', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--01a19ffa-cb01-712c-ab40-d32b774b720e-0' tool_calls=[{'name': 'get_yf_stock_history', 'args': {'stock_history_input': {'ticker': 'TSLA', 'period': '1mo'}}, 'id': 'call_uMOhEmy0efzNTb6mm05TiX4V', 'type': 'tool_call'}, {'name': 'get_yf_stock_history', 'args': {'stock_history_input': {'ticker': 'TSLA', 'period':

In [10]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]
    print(tool_call["args"])
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)
    print(tool_msg)

{'stock_history_input': {'ticker': 'TSLA', 'period': '1mo'}}
content='| Date                      |   Open |   High |    Low |   Close |      Volume |   Dividends |   Stock Splits |\n|:--------------------------|-------:|-------:|-------:|--------:|------------:|------------:|---------------:|\n| 2026-10-05 00:00:00-04:00 | 369    | 381.59 | 364.91 |  378.73 | 4.22327e+07 |           0 |              0 |\n| 2026-10-06 00:00:00-04:00 | 381.98 | 383.33 | 378.52 |  380.68 | 2.77354e+07 |           0 |              0 |\n| 2026-10-07 00:00:00-04:00 | 378.45 | 382.35 | 374.43 |  377.81 | 2.55946e+07 |           0 |              0 |\n| 2026-10-08 00:00:00-04:00 | 374.12 | 375.96 | 368.03 |  375    | 2.83949e+07 |           0 |              0 |\n| 2026-10-09 00:00:00-04:00 | 382.38 | 388.56 | 379.71 |  382.7  | 3.89524e+07 |           0 |              0 |\n| 2026-10-12 00:00:00-04:00 | 382.61 | 382.7  | 377.58 |  378.03 | 2.02443e+07 |           0 |              0 |\n| 2026-10-13 00:00:00-04:0

In [11]:
LLM_with_tools.invoke(messages)

AIMessage(content='테슬라(TSLA)의 주가는 한 달 전과 비교할 때 하락했습니다. \n\n- **한 달 전 (2026년 10월 2일)** 주가는 **378.73** 달러였습니다.\n- **현재 (2026년 11월 2일)** 주가는 **403.82** 달러입니다.\n\n결론적으로, 테슬라의 주가는 한 달 전보다 상승하였습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 92, 'prompt_tokens': 1769, 'total_tokens': 1861, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_fbfbfe4157', 'id': 'chatcmpl-Eg1nJJKhJJDS0jG7K0a5ZZFWnmwuR', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a19ffa-e038-7062-9cd2-9384b229c450-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1769, 'output_tok